In [1]:
# 0: Lib
import numpy as np
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
from pathlib import Path
from torchvision import transforms
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from tqdm import tqdm
import time
import os
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from torch.utils.data import DataLoader, Subset, random_split



In [2]:

# 1: PersonalGazeDataset
# Lädt Bilder als PIL-Objekte und transformiert sie NICHT vorab.

class PersonalGazeDataset(Dataset):

    def __init__(
        self,
        root_dir,
        dataset_size=None,
        ##### orginal: ###############
        # dataset_name="norm_labels.csv"

        ##### clear: ###############
        dataset_name="norm_labels_clean.csv"
        
    ):
        self.root_dir = Path(root_dir)

        # CSV mit Bildnamen und normalisierten Blickkoordinaten
        csv_path = self.root_dir / dataset_name
        self.df = pd.read_csv(csv_path)

        # Prüfen, ob die erforderlichen Spalten existieren
        required_columns = {"frame", "x", "y"}

        if not required_columns.issubset(self.df.columns):
            raise ValueError(
                f"CSV muss diese Spalten enthalten: {required_columns}"
            )

        # Optional: Dataset für Tests verkleinern
        if dataset_size is not None:
            if dataset_size <= 0:
                raise ValueError("dataset_size muss > 0 sein.")

            self.df = self.df.iloc[:dataset_size].copy()

        if self.df.empty:
            raise ValueError("Das Dataset ist leer.")

        # Prüfen, ob Labels numerisch und gültig sind
        self.df[["x", "y"]] = self.df[["x", "y"]].apply(
            pd.to_numeric,
            errors="raise"
        )

        if not np.isfinite(self.df[["x", "y"]].to_numpy()).all():
            raise ValueError("Die Labels enthalten NaN oder Inf.")

    def __len__(self):
        return len(self.df)

    def get_raw_item(self, idx):
        """
        Gibt ein unverändertes PIL-Bild und die Ground-Truth zurück.
        Hier findet KEINE Transformation statt.
        """

        row = self.df.iloc[idx]

        image_path = self.root_dir / "images2" / str(row["frame"])

        # Image.open innerhalb von with sicher schließen
        with Image.open(image_path) as img:
            image = img.convert("RGB")

        target = torch.tensor(
            [row["x"], row["y"]],
            dtype=torch.float32
        )

        return image, target

    def __getitem__(self, idx):
        """
        Standardzugriff ohne Transformation.
        Die Transformation wird im TransformSubset angewendet.
        """
        return self.get_raw_item(idx)

In [ ]:
# # 1: calss PersonalGazeDataset
# class PersonalGazeDataset(Dataset):

#     def __init__(
#         self,
#         root_dir,
#         transform=None,
#         dataset_size=None,
#         read_all4once=True
#     ):

#         self.root_dir = Path(root_dir)
#         self.transform = transform
#         self.read_all4once = read_all4once

#         self.df = pd.read_csv(
#             self.root_dir / "norm_labels.csv"
#             # self.root_dir 
#         )

#         self.dataset_size = (
#             dataset_size
#             if dataset_size is not None
#             else len(self.df)
#         )


#         if self.read_all4once:

#             # Form des transformierten Bildes bestimmen
#             img = Image.new("RGB", (500, 300))
#             out = transform(img)

#             self.images = torch.zeros(
#                 [self.dataset_size] + list(out.shape)
#             )

#             self.targets = torch.zeros(
#                 self.dataset_size,
#                 2
#             )


#             for idx in tqdm(range(self.dataset_size)):

#                 row = self.df.iloc[idx]

#                 image = Image.open(
#                     self.root_dir
#                     / "images"
#                     / row["frame"]
#                 ).convert("RGB")


#                 self.targets[idx] = torch.tensor(
#                     [
#                         row["x"],
#                         row["y"]
#                     ],
#                     dtype=torch.float32
#                 )


#                 if self.transform:
#                     self.images[idx] = self.transform(image)
#                 else:
#                     self.images[idx] = image

#     def __len__(self):

#         return self.dataset_size
        

#     def get_raw_item(self, idx):
    
#         image = self.images[idx].clone()
#         target = self.targets[idx].clone()
    
#         return image, target


#     # def __getitem__(self, idx):

#     #     if self.read_all4once:

#     #         return (
#     #             self.images[idx],
#     #             self.targets[idx]
#     #         )

#     #     row = self.df.iloc[idx]

#     #     image = Image.open(
#     #         self.root_dir
#     #         / "images"
#     #         / row["frame"]
#     #     ).convert("RGB")

#     #     target = torch.tensor(
#     #         [
#     #             row["x"],
#     #             row["y"]
#     #         ],
#     #         dtype=torch.float32
#     #     )

#     #     if self.transform:
#     #         image = self.transform(image)

#     #     return image, target

#     def __getitem__(self, idx):

#         image, target = self.get_raw_item(idx)
    
#         if self.transform:
#             image = self.transform(image)
    
#         return image, target

In [3]:

class TransformSubset(Dataset):

    def __init__(
        self,
        dataset,
        indices,
        transform=None
    ):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        original_idx = self.indices[idx]

        # PIL-Bild und Label aus dem Basis-Dataset
        image, target = self.dataset.get_raw_item(original_idx)

        # Transformation GENAU EINMAL anwenden
        if self.transform is not None:
            image = self.transform(image)

        return image, target

In [4]:
# 2: func diagonal_error
def diagonal_errors(model, loader, device):

    # Evaluation-Modus
    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets_all = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)

    mae = mean_absolute_error(targets_all, predictions)

    rmse = np.sqrt(mean_squared_error(targets_all, predictions))

    diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    return mae, rmse, diagonal_error_pct

In [5]:
def evaluate_model(model, loader, device):

    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    return targets, predictions, errors

In [6]:
def sensitive_loss(preds, targets):

    x_total = 0.0

    for i in range(len(targets)):

        x = criterion_base(preds[i], targets[i])

        distance_from_center = (
            torch.abs(targets[i][0] - 0.5) +
            torch.abs(targets[i][1] - 0.5)
        )

        weight = 1.0 + distance_from_center

        x_total += x * weight

    return x_total / len(targets)

In [12]:

def create_splits(
    full_dataset,
    train_size,
    validat_size,
    test_size,
    split_type="random",
    seed=42,
    original_fps=60,
    frame_step=5,
    reset_interval=5,
    exclusion_time=0.5,
    test_dataset=None
):

    dataset_size = len(full_dataset)


    # ============================================================
    # A: Random Split
    # ============================================================

    if split_type == "random":

        generator = torch.Generator().manual_seed(seed)

        train_dataset, validat_dataset, test_dataset = random_split(
            full_dataset,
            [train_size, validat_size, test_size],
            generator=generator
        )



    # ============================================================
    # B1: Sequential Split
    # ============================================================

    elif split_type == "sequential":

        train_indices = list(
            range(0, train_size)
        )

        validat_indices = list(
            range(
                train_size,
                train_size + validat_size
            )
        )

        test_indices = list(
            range(
                train_size + validat_size,
                dataset_size
            )
        )

        train_dataset = Subset(
            full_dataset,
            train_indices
        )

        validat_dataset = Subset(
            full_dataset,
            validat_indices
        )

        test_dataset = Subset(
            full_dataset,
            test_indices
        )



    # ============================================================
    # B2: Test first, then Train + Validation
    # ============================================================

    elif split_type == "test_first_random":

        generator = torch.Generator().manual_seed(seed)

        train_valid_size = train_size + validat_size

        train_valid_dataset, test_dataset = random_split(
            full_dataset,
            [train_valid_size, test_size],
            generator=generator
        )

        train_dataset, validat_dataset = random_split(
            train_valid_dataset,
            [train_size, validat_size],
            generator=generator
        )


    # ============================================================
    # C: Video 1 = Train + Validation
    #    Video 2 = Test
    # ============================================================

    elif split_type == "different_videos":

        if test_dataset is None:

            raise ValueError(
                "Für split_type='different_videos' "
                "muss test_dataset angegeben werden."
            )


        # --------------------------------------------------------
        # Train + Validation aus Video 1
        # --------------------------------------------------------

        train_valid_size = train_size + validat_size

        if train_valid_size > len(full_dataset):

            raise ValueError(
                "Train + Validation sind größer als "
                "das Train/Validation-Video."
            )


        generator = torch.Generator().manual_seed(seed)


        train_dataset, validat_dataset = random_split(
            full_dataset,
            [train_size, validat_size],
            generator=generator
        )


        # --------------------------------------------------------
        # Test = komplettes zweites Video
        # --------------------------------------------------------

        test_dataset = test_dataset


        print("\n" + "=" * 60)
        print("DIFFERENT-VIDEO SPLIT")
        print("=" * 60)

        print(
            f"Train/Validation Video: "
            f"{len(full_dataset)} Samples"
        )

        print(
            f"Test Video: "
            f"{len(test_dataset)} Samples"
        )

        print("\nSplit:")

        print(
            f"Train:       {len(train_dataset)}"
        )

        print(
            f"Validation:  {len(validat_dataset)}"
        )

        print(
            f"Test:        {len(test_dataset)}"
        )

        print("=" * 60)


    # ============================================================
    # D: Reset-aware Random Split
    # ============================================================

    elif split_type == "reset_aware":

        # --------------------------------------------------------
        # Tatsächliche Samplingrate nach Frame-Subsampling
        # --------------------------------------------------------

        effective_fps = original_fps / frame_step


        # --------------------------------------------------------
        # Anzahl gespeicherter Samples pro Reset-Intervall
        # --------------------------------------------------------

        reset_samples = int(
            round(reset_interval * effective_fps)
        )

        if reset_samples <= 0:
            raise ValueError(
                "reset_samples muss größer als 0 sein."
            )


        # --------------------------------------------------------
        # Anzahl gespeicherter Samples, die nach jedem Reset
        # ausgeschlossen werden
        # --------------------------------------------------------

        exclusion_samples = int(
            round(exclusion_time * effective_fps)
        )


        print("\n" + "=" * 60)
        print("RESET-AWARE SPLIT")
        print("=" * 60)

        print(f"Original FPS:          {original_fps}")
        print(f"Frame Step:            {frame_step}")
        print(f"Effektive FPS:         {effective_fps}")

        print(f"\nReset alle:            {reset_interval} s")
        print(f"Reset-Samples:         {reset_samples}")

        print(f"\nAusschlusszeit:        {exclusion_time} s")
        print(f"Ausschluss-Samples:    {exclusion_samples}")


        # --------------------------------------------------------
        # Gültige und ausgeschlossene Indizes bestimmen
        # --------------------------------------------------------

        valid_indices = []
        excluded_indices = []


        for idx in range(dataset_size):

            position_in_reset = idx % reset_samples


            # Erste 0.5 Sekunden nach jedem Reset
            if position_in_reset < exclusion_samples:

                excluded_indices.append(idx)

            else:

                valid_indices.append(idx)


        print(f"\nGesamte Samples:       {dataset_size}")
        print(f"Ausgeschlossen:        {len(excluded_indices)}")
        print(f"Verwendbar:            {len(valid_indices)}")


        # --------------------------------------------------------
        # Prüfen, ob genügend Daten vorhanden sind
        # --------------------------------------------------------

        required_size = (
            train_size
            + validat_size
            + test_size
        )


        if required_size > len(valid_indices):

            raise ValueError(
                "\nNicht genügend gültige Samples!\n"
                f"Benötigt:   {required_size}\n"
                f"Verfügbar: {len(valid_indices)}"
            )


        # --------------------------------------------------------
        # Gültige Daten zufällig mischen
        # --------------------------------------------------------

        generator = torch.Generator().manual_seed(seed)

        permutation = torch.randperm(
            len(valid_indices),
            generator=generator
        ).tolist()


        shuffled_indices = [
            valid_indices[i]
            for i in permutation
        ]


        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------

        train_indices = shuffled_indices[
            :train_size
        ]


        # --------------------------------------------------------
        # Validation
        # --------------------------------------------------------

        validat_indices = shuffled_indices[
            train_size:
            train_size + validat_size
        ]


        # --------------------------------------------------------
        # Test
        # --------------------------------------------------------

        test_indices = shuffled_indices[
            train_size + validat_size:
            train_size + validat_size + test_size
        ]


        # --------------------------------------------------------
        # Subsets
        # --------------------------------------------------------

        train_dataset = Subset(
            full_dataset,
            train_indices
        )

        validat_dataset = Subset(
            full_dataset,
            validat_indices
        )

        test_dataset = Subset(
            full_dataset,
            test_indices
        )


        print("\nSplit:")
        print(f"Train:                 {len(train_dataset)}")
        print(f"Validation:            {len(validat_dataset)}")
        print(f"Test:                  {len(test_dataset)}")

        print("=" * 60)


    else:

        raise ValueError(
            f"Unbekannter split_type: {split_type}"
        )


    return (
        train_dataset,
        validat_dataset,
        test_dataset
    )

In [ ]:
# # Transformationen:
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),

#     transforms.ColorJitter(
#         brightness=0.3,
#         contrast=0.3,
#         saturation=0.2
#     ),

#     transforms.RandomGrayscale(p=0.05),

#     transforms.GaussianBlur(
#         kernel_size=3,
#         sigma=(0.1, 1.5)
#     ),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])

In [7]:
# ================================================
# Train Transform:
# ================================================

# Training: Augmentation + Normalisierung
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



# ================================================
# Test/Validation Transform:
# ================================================

# Validation und Test: nur Resize + Normalisierung
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
# Hypoparameter:

name_dataset_type = ['norm_labels_clean.csv','norm_labels.csv', 'labels.csv']
dataset_size_type = [[10000, 2000], [1000, 200]]
dataset_type = ["norm_subject", "norm_random"]
batch_size_type = [32, 64, 128]
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
epochs_num = [1, 2, 10, 25, 500]



optimizer_name = "AdamW"

# ##########################################################################
# 1.type Dataset: 'norm_labels.csv' or 'labels.csv'
dataset_name = name_dataset_type[0]                       # norm_labels_clean.csv
# dataset_name = name_dataset_type[1]                       # norm_labels.csv
# dataset_name = name_dataset_type[2]                       # labels.csv
print(f"\n dataset_name: \t\t {dataset_name}")


# 2.dataset_sizt: (10000 & 2000) or (1000, 200)
# def_dataset_size = dataset_size_type[0]                   # [10000, 2000]
def_dataset_size = dataset_size_type[1]                   # [1000, 200]

print(f"\n def_dataset_size: \n train: \t\t {def_dataset_size[0]} \n test: \t\t\t {def_dataset_size[1]}")


# 3.batch_size: '32', '64' or '128'
batch_size = batch_size_type[0]                           # 32
# batch_Size = batch_size_type[1]                           # 64
# batch_Size = batch_size_type[2]                           # 128
print(f"\n batch_Size: \t\t {batch_size}")


# 4.type of dataset-split: "norm_subject_independed" or "norm_random"
# load model:
dataset_type = ["norm_subject", "norm_random"]

# load model:
#============================================================================
#                 normalize_subject_indipended 
#============================================================================
def_dataset = dataset_type[0]                             # norm_subject
# saved_model = "./models/best_models/best_ResNet_Sigmoid_norm_subject.path"


#============================================================================
#                 normalize_random
#============================================================================
# # def_dataset = dataset_type[1]                             # norm_random
# saved_model = "./models/best_models/best_ResNet_Sigmoid_norm_random.path"

print(f"\n def_dataset: \t\t {def_dataset}")

# load model:



# 5.learning_rate: '1e-3', '1e-4', '1e-5' or '1e-6'
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
# learning_rate = lr_type[0]                                # 1e-3
learning_rate = lr_type[1]                                # 1e-4
# learning_rate = lr_type[2]                                # 1e-5
# learning_rate = lr_type[3]                                # 1e-6
print(f"\n learning_rate: \t {learning_rate}")


# 6.number of epochs: 1, 2, 10, 25 or 500
epochs = epochs_num[-1]                                   # 500
print(f"\n epochs: \t\t {epochs}")




weight_Decay = 1e-5

active_func = None

patience = 3               # after 3 Epochen without Optimierung has to stop training
print(f"\n patience: \t\t {patience}")





# session = "03"

# best_model_name = "./models/ResNet_best_optim-model_norm_subject_1000-200.path"

# last best model: with Sigmoid
# saved_model = "./models/best_models/best_ResNet_Sigmoid.path"


# load model:



best_saved_model = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"
print(f"\n model_name: \t\t {best_saved_model}")


#============================================================================
#                 normalize_subject_indipended 
#============================================================================
# saved_model1 = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"

#============================================================================
#                 normalize_random
#============================================================================
# saved_model2 = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"



 dataset_name: 		 norm_labels_clean.csv

 def_dataset_size: 
 train: 		 1000 
 test: 			 200

 batch_Size: 		 32

 def_dataset: 		 norm_subject

 learning_rate: 	 0.0001

 epochs: 		 500

 patience: 		 3

 model_name: 		 ./models/best_models/best_ResNet_Sigmoid_norm_subject.path


In [10]:
# 9: ResNet18 (Pretrainiertes Modell):
# Load:
def reset_model(act=None):
    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model_name = model.__class__.__name__
    
    # ---------------------------------------
    # Activation function for the last layer
    # ---------------------------------------
    if act == "Sigmoid":
        last_layer = nn.Sigmoid()

    elif act == "Gaussian":
        last_layer = GaussianActivation(sigma=1.0)

    elif act == "ReLU":
        last_layer = nn.ReLU()

    elif act == "None":
        last_layer = nn.Identity()

    else:
        raise ValueError(
            f"Unknown activation function: {act}"
        )

    # ---------------------------------------
    # Replace original ResNet FC
    # ---------------------------------------
    model.fc = nn.Sequential(
        nn.Linear(
            model.fc.in_features,
            512
        ),
        nn.ReLU(),

        nn.Linear(
            512,
            256
        ),
        nn.ReLU(),

        nn.Dropout(0.2),

        nn.Linear(
            256,
            128
        ),
        nn.ReLU(),

        nn.Linear(
            128,
            2
        ),

        # Last activation
        last_layer
    )

    return model, model_name

In [11]:
# 10: GPU or CPU
# check:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [12]:
norm_labels_files = []
root_folder = Path("./personalization/")


for folder in sorted(root_folder.iterdir()):

    if folder.is_dir():

        norm_file = folder / "norm_labels.csv"


        if not norm_file.exists():

            # print(f"Fehlt: {norm_file.resolve()}\n\n")
            continue

        else:

            norm_labels_files.append(norm_file.as_posix())


for n in norm_labels_files:
    print(f"norm_file: {n}")

print(f"\n Anzahl der norm_files: {len(norm_labels_files)}\n")


sessions = []

for i in range(len(norm_labels_files)):
    path = str(norm_labels_files[i])
    session = path.split("/")[1]
    # print(f"Session: {session}")
    sessions.append(session)

print(f"Sessions: {sessions}")

print(f"\n Anzahl der Sessions: {len(sessions)}")

norm_file: personalization/01/norm_labels.csv
norm_file: personalization/02/norm_labels.csv
norm_file: personalization/03/norm_labels.csv
norm_file: personalization/04/norm_labels.csv
norm_file: personalization/05/norm_labels.csv
norm_file: personalization/06/norm_labels.csv
norm_file: personalization/07/norm_labels.csv
norm_file: personalization/08/norm_labels.csv
norm_file: personalization/09/norm_labels.csv
norm_file: personalization/10/norm_labels.csv
norm_file: personalization/11/norm_labels.csv
norm_file: personalization/12/norm_labels.csv
norm_file: personalization/13/norm_labels.csv
norm_file: personalization/14/norm_labels.csv
norm_file: personalization/15/norm_labels.csv
norm_file: personalization/16/norm_labels.csv
norm_file: personalization/17/norm_labels.csv
norm_file: personalization/18/norm_labels.csv
norm_file: personalization/19/norm_labels.csv
norm_file: personalization/20/norm_labels.csv
norm_file: personalization/21/norm_labels.csv
norm_file: personalization/22/norm

In [13]:

# ============================================================
# Smoke Test: Dataset und DataLoader prüfen
# ============================================================
session = '25'
print(f"\nSession: {session}")

dirs = f"personalization/{session}"

csv_path = Path(dirs) / "norm_labels_clean.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Fehlt: {csv_path.resolve()}")

# print(f"csv_pfad: {csv_path}")

df = pd.read_csv(csv_path)


dataset_size = len(df)

print(f"Gesamte Daten: {dataset_size}")


# ============================================================
# Full Dataset
# ============================================================

# full_dataset = PersonalGazeDataset(
#     root_dir=dirs,
#     transform=None,
# )

full_dataset = PersonalGazeDataset(
    root_dir=dirs,
    dataset_name="norm_labels_clean.csv"
)

dataset_size = len(full_dataset)

print(f"Dataset Size: {dataset_size}")



# 1. Rohbild muss ein PIL-Bild sein
raw_image, raw_target = full_dataset.get_raw_item(0)

assert isinstance(raw_image, Image.Image), (
    f"Erwartet PIL.Image, erhalten: {type(raw_image)}"
)

print("Raw image type:", type(raw_image))
print("Raw target:", raw_target)


# ============================================================
# 1. Reproduzierbare Random-Split-Indizes
# ============================================================

split_seed = 42

generator = torch.Generator().manual_seed(split_seed)

indices = torch.randperm(
    dataset_size,
    generator=generator
).tolist()

# ============================================================
# Split Sizes
# ============================================================

train_size = int(dataset_size * 0.70)
validat_size = int(dataset_size * 0.15)

test_size = (
    dataset_size
    - train_size
    - validat_size
)

train_indices = indices[:train_size]

validat_indices = indices[
    train_size:train_size + validat_size
]

test_indices = indices[
    train_size + validat_size:
]


# ============================================================
# 2. Subsets mit getrennten Transformationen
# ============================================================

train_dataset = TransformSubset(
    full_dataset,
    train_indices,
    transform=train_transform
)

validat_dataset = TransformSubset(
    full_dataset,
    validat_indices,
    transform=eval_transform
)

test_dataset = TransformSubset(
    full_dataset,
    test_indices,
    transform=eval_transform
)


# ============================================================
# 3. Kontrolle der Dataset-Größen
# ============================================================

print(f"Train:      {len(train_dataset)}")
print(f"Validation: {len(validat_dataset)}")
print(f"Test:       {len(test_dataset)}")


# ============================================================
# 4. DataLoader
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    persistent_workers=True
)



validat_loader = DataLoader(
    validat_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    persistent_workers=True
)



test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    persistent_workers=True
)

##########################################################################
##########################################################################
train_eval_dataset = TransformSubset(
    full_dataset,
    train_indices,
    transform=eval_transform
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    persistent_workers=True
)



# 2. Transformiertes Bild muss ein Tensor sein
train_image, train_target = train_dataset[0]

assert isinstance(train_image, torch.Tensor)

assert train_image.shape == (3, 224, 224)

assert train_target.shape == (2,)

print("Train image shape:", train_image.shape)
print("Train target:", train_target)


# 3. Validation und Test ebenfalls prüfen
val_image, val_target = validat_dataset[0]
test_image, test_target = test_dataset[0]

assert isinstance(val_image, torch.Tensor)
assert isinstance(test_image, torch.Tensor)

assert val_image.shape == (3, 224, 224)
assert test_image.shape == (3, 224, 224)


# 4. Einen echten Batch laden
images, targets = next(iter(train_loader))

assert images.ndim == 4
assert images.shape[1:] == (3, 224, 224)

assert targets.ndim == 2
assert targets.shape[1] == 2

print("Batch images:", images.shape)
print("Batch targets:", targets.shape)

print("\nSmoke Test erfolgreich!")


Session: 25
Gesamte Daten: 720
Dataset Size: 720
Raw image type: <class 'PIL.Image.Image'>
Raw target: tensor([0.0944, 0.9557])
Train:      503
Validation: 108
Test:       109
Train image shape: torch.Size([3, 224, 224])
Train target: tensor([0.1023, 0.4537])
Batch images: torch.Size([32, 3, 224, 224])
Batch targets: torch.Size([32, 2])

Smoke Test erfolgreich!


In [31]:

results = {}

# sess = ['01', '02']
# sess = ['01']

# for s in sess: 
for s in sessions:

    session = s

    print("\n" + "=" * 70)
    print(f"{def_dataset.upper()} Session: {session.upper()}")
    print("=" * 70)
    
    dirs = f"personalization/{session}"
    
    csv_path = Path(dirs) / "norm_labels.csv"
    
    if not csv_path.exists():
        raise FileNotFoundError(f"Fehlt: {csv_path.resolve()}")
    
    print(f"csv_pfad: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    dataset_size = len(df)
    
    print(f"Gesamte Daten: {dataset_size}")
    
    
    # ============================================================
    # Full Dataset
    # ============================================================
    
    # full_dataset = PersonalGazeDataset(
    #     root_dir=dirs,
    #     transform=None,
    # )

    full_dataset = PersonalGazeDataset(
        root_dir=dirs,
        dataset_name="norm_labels.csv"
    )

    dataset_size = len(full_dataset)

    print(f"Dataset Size: {dataset_size}")
    

    # ============================================================
    # 1. Reproduzierbare Random-Split-Indizes
    # ============================================================

    split_seed = 42

    generator = torch.Generator().manual_seed(split_seed)

    indices = torch.randperm(
        dataset_size,
        generator=generator
    ).tolist()

    # ============================================================
    # Split Sizes
    # ============================================================
    
    train_size = int(dataset_size * 0.70)
    validat_size = int(dataset_size * 0.15)
    
    test_size = (
        dataset_size
        - train_size
        - validat_size
    )

    train_indices = indices[:train_size]

    validat_indices = indices[
        train_size:train_size + validat_size
    ]

    test_indices = indices[
        train_size + validat_size:
    ]


    # ============================================================
    # 2. Subsets mit getrennten Transformationen
    # ============================================================

    train_dataset = TransformSubset(
        full_dataset,
        train_indices,
        transform=train_transform
    )

    validat_dataset = TransformSubset(
        full_dataset,
        validat_indices,
        transform=eval_transform
    )

    test_dataset = TransformSubset(
        full_dataset,
        test_indices,
        transform=eval_transform
    )


    # ============================================================
    # 3. Kontrolle der Dataset-Größen
    # ============================================================

    print(f"Train:      {len(train_dataset)}")
    print(f"Validation: {len(validat_dataset)}")
    print(f"Test:       {len(test_dataset)}")


    # ============================================================
    # 4. DataLoader
    # ============================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    validat_loader = DataLoader(
        validat_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )

##########################################################################
##########################################################################
    train_eval_dataset = TransformSubset(
        full_dataset,
        train_indices,
        transform=eval_transform
    )

    train_eval_loader = DataLoader(
        train_eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )
##########################################################################
##########################################################################




    ##########################################################################
    ##########################################################################

    # ========================================================================
    # Data Loading Strategies:
    # ========================================================================

    # # A1: Random
    # #split = "a_random"
    # train_dataset, validat_dataset, test_dataset = create_splits(
    #     full_dataset,
    #     train_size,
    #     validat_size,
    #     test_size,
    #     split_type="random",
    #     seed=42
    # )
    
    
    # print(f"Train:      {len(train_dataset)}")
    # print(f"Validation: {len(validat_dataset)}")
    # print(f"Test:       {len(test_dataset)}")



    
    
    
    # # Loader:
    # train_loader = DataLoader(
    #     train_dataset,
    #     batch_size=train_size,
    #     shuffle=True,
    #     num_workers=8,
    #     persistent_workers=True
    # )
    
    
    
    # validat_loader = DataLoader(
    #     validat_dataset,
    #     batch_size=validat_size,
    #     shuffle=False,
    #     num_workers=8,
    #     persistent_workers=True
    # )
    
    
    
    # test_loader = DataLoader(
    #     test_dataset,
    #     batch_size=test_size,
    #     shuffle=False,
    #     num_workers=8,
    #     persistent_workers=True
    # )

    ##########################################################################
    ##########################################################################


    # ============================================================
    # Training:
    # ============================================================
    
    # acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']
    act = "Sigmoid"
    
    trainable_layers = "FC only"
    
    
    print("\n" + "=" * 70)
    print(f"{def_dataset} EXPERIMENT: ")
    
    
    print(f"\t Dataset Type:         {dataset_name}")
    print(f"\t Dataset Split:        {def_dataset}")
    print(f"\t Train Size:           {def_dataset_size[0]}")
    print(f"\t Test Size:            {def_dataset_size[1]}")
    print(f"\t Batch Size:           {batch_size}")
    print(f"\t Learning Rate:        {learning_rate}")
    print(f"\t Activation:           {act}")
    print(f"\t Trainable Layers:     {trainable_layers}")
    print(f"\t Loaded_Model:         {best_saved_model}")
    print(f"\t Max_Epochs:           {epochs}")
    print(f"\t Patience:             {patience}")
    print("=" * 70)
    
    
    
    
    if globals().get("bas_const_err") is None:
        bas_const_err = 27.245
    
    if globals().get("bas_rand_err") is None:
        bas_rand_err = 38.961
    
    
    
    # --------------------------------------------------------
    # Start total training timer
    # --------------------------------------------------------
    
    train_start = time.perf_counter()
    
    
    
    epochs = epochs
    
    
    
    model_output = './models/best_models/personal/'
    model_dir= Path(model_output)
    
    if not os.path.exists(model_dir):
        model_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    diagrams_output = './results/best_diagrams/personal'
    diagrams_dir= Path(diagrams_output)
    
    if not os.path.exists(diagrams_dir):
        diagrams_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    
    # ============================================================
    # Store train and test error curves
    # ============================================================
    
    # train_errors = []
    # test_errors = []
    
    best_error = float("inf")
    
    patience_counter = 0                        # Number Epochen without Optimierung
    improvements = 0                            # Number Epochen with Optimierung
    best_epoch = 0
    
    diag_train_errors = []
    diag_validat_errors  = []
    
    act_time_epochs = {}
    
    
    
    criterion = nn.L1Loss()
    
    criterion_base = nn.L1Loss()
    
    # --------------------------------------------------------
    # Reset model
    # --------------------------------------------------------
    
    model, model_name = reset_model(act=act)

    checkpoint = torch.load(
        # checkpoint_path,
        best_saved_model,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    # --------------------------------------------------------
    # Freeze all parameters
    # --------------------------------------------------------
    
    for param in model.parameters():
        param.requires_grad = False
    
    
    # --------------------------------------------------------
    # Unfreeze ONLY FC
    # --------------------------------------------------------
    
    for param in model.layer4.parameters():
        param.requires_grad=False
    
    
    for param in model.fc.parameters():
        param.requires_grad = True
    
    
    
    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_Decay
    )
    
    
    
    
    
    print('\n ',"=#=" * 25)
    print(f"\t\t Session: {session}")
    print(' ',"=#=" * 25)
    
    
    # ============================================================
    # Initial evaluation before training
    # ============================================================
    print("\n\t Initial evaluation: \n")
    
    model.eval()
    
    print("\n Train_error:")
    # train_mae, train_rmse, train_diag_pct = diagonal_errors(
    #     model,
    #     train_loader,
    #     device
    # )

    train_mae, train_rmse, train_diag_pct = diagonal_errors(
        model,
        train_eval_loader,
        device
    )
    
    train_diag_pct = np.round(float(train_diag_pct), 4)
    # diag_train_errors.append(train_diag_pct)
    
    
    
    
    print("\n Validat_error:")
    validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
        model,
        validat_loader,
        device
    )
    
    validat_diag_pct = np.round(float(validat_diag_pct), 4)
    # diag_validat_errors.append(validat_diag_pct)
    
    
    
    print(
        f"Epoch 0 | "
        f"Train error: {train_diag_pct:.4f}% | "
        f"Validat error: {validat_diag_pct:.4f}%"
    )
    
    
    
    # Initial best test error
    if validat_diag_pct < best_error:  
        best_error = validat_diag_pct
        best_epoch = 0
    
    
    # save the better Modell
    torch.save(
        model.state_dict(),
        f"{model_output}"
        f"best_{model_name}_{act}_{def_dataset}_{session}.path"
    )
    
    # ============================================================
    # Training
    # ============================================================
    
    print("\n\n\t Training: \n")
    
    for epoch in range(epochs):
    
        epoch_start = time.perf_counter()
    
        model.train()
    
        running_loss = 0.0
    
    
        loop = tqdm(
            train_loader,
            desc=f"Normalized | Epoch {epoch + 1}"
        )
        
    
        for images, targets in loop:
            
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            preds = model(images)
    
            ################################
            # Loss 1:
            ################################
            
            # loss = criterion(
            #     preds,
            #     targets
            # )
    
    
            
            ################################
            # Loss 2: 
            ################################
            
            loss = sensitive_loss(
                preds,
                targets
            )
            
            ################################
    
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
            loop.set_postfix(
                loss=loss.item()
            )
    
        
    
        # ========================================================
        # Evaluation after epoch
        # ========================================================
    
        model.eval()
    
    
        # --------------------------------------------------------
        # Train error
        # --------------------------------------------------------
        print("\n Train:")
        # train_mae, train_rmse, train_diag_pct = diagonal_errors(
        #     model,
        #     train_loader,
        #     device
        # )

        train_mae, train_rmse, train_diag_pct = diagonal_errors(
            model,
            train_eval_loader,
            device
        ) 
   
        
    
        train_diag_pct = np.round(float(train_diag_pct), 4)
        diag_train_errors.append(train_diag_pct)
    
    
        # --------------------------------------------------------
        # Test error
        # --------------------------------------------------------
        print("\n Validat:")
        validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
            model,
            validat_loader,
            device
        )
    
        validat_diag_pct = np.round(float(validat_diag_pct), 4)
        diag_validat_errors.append(validat_diag_pct)
    
    
        # --------------------------------------------------
        # Early Stopping
        # --------------------------------------------------
    
        if validat_diag_pct < best_error:
    
            best_error = validat_diag_pct
    
            best_epoch = epoch + 1
    
            patience_counter = 0
            improvements += 1
    
            print(
                f"\nValidat-Diagonal-Error: "
                f"{validat_diag_pct:.4f}% | "
                f"Improvement: {improvements}"
            )
    
            # save the better Modell
            torch.save(
                model.state_dict(),
                f"{model_output}"
                f"best_{model_name}_{act}_{def_dataset}_{session}.path"
            )
        
    
        else:
    
            # without imporovement
            patience_counter += 1
    
            print(
                f"\nPatience: "
                f"{patience_counter}/{patience}"
            )
    
    
            if patience_counter >= patience:
    
                print(
                    f"\nEarly Stopping after "
                    f"{epoch + 1} epochs."
                )
                print(
                    f"Imporovments totally: {improvements}"
                )
    
                break
        
    
        # ========================================================
        # Epoch information
        # ========================================================
    
        epoch_end = time.perf_counter()
    
        epoch_time = epoch_end - epoch_start
    
    
        print(
            f"\n[{datetime.now().strftime('%H:%M:%S')}] "
            f"Epoch {epoch + 1}: | " 
            f"{epoch_time:.2f} s | "
            f"Loss: {running_loss / len(train_loader):.4f} | "
            f"Train-Error: {train_diag_pct:.4f}% | "
            f"Test-Error: {validat_diag_pct:.4f}%"
        )
    
        torch.save(
            model.state_dict(),
            f"{model_output}"
            f"last_{model_name}_{act}_{def_dataset}_{session}.path"
        )
    
    
    
    # ============================================================
    # Total running time
    # ============================================================
    
    train_end = time.perf_counter()
    
    elapsed_running_time = train_end - train_start
    
    elapsed_minutes = elapsed_running_time / 60
    
    print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} s")
    print(f"totll running-tiems (min): {elapsed_minutes:.2f} min\n")
    
    # ============================================================
    # Number of completed epochs
    # ============================================================
    
    epochs_completed = len(diag_validat_errors)
    
    
    # # ============================================================
    # # Store normalized-dataset results
    # # ============================================================
    
    results[session] = {
        "split_type" : def_dataset,
        "train_errors": diag_train_errors,
        "validat_errors": diag_validat_errors,
        "best_validat_error": best_error,
        "running_time_minutes": elapsed_minutes,
        "epochs": epochs_completed,
        "best_epoch": best_epoch,
        "improvements": improvements
    }
    
    
    # ========================================================
    # Print result for this learning rate
    # ========================================================
    
    print("\n" + "=" * 70)
    print(f"{def_dataset.upper()} Session: {session.upper()} RESULTS: ")
    print("=" * 70)
    
    print(
        f"Best Validation Error:   {results[session]['best_validat_error']:.4f}%"
    )
    print(
        f"Running Time:            {results[session]['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                  {results[session]['epochs']}"
    )
    print(
        f"Best Epoch:              {results[session]['best_epoch']}"
    )
    print(
        f"Improvements:            {results[session]['improvements']}"
    )
    print("=" * 70)
    
    
    # ---------- Evaluation ----------
    
    # norm_subject:
    # saved_model1 = "./models/best_models/best_ResNet_Sigmoid_norm_subject.path"
    # saved_model1 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"
    
    # norm_random:
    # saved_model2 = "./models/best_models/best_ResNet_Sigmoid_norm_random.path"
    # saved_model2 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"

    test_saved_model = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"

    checkpoint = torch.load(
        test_saved_model,
        map_location=device
    )
    
    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    model.eval()
    
    
    print(f"Test:\n")
    test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
    
    test_diag_pct = np.round(float(test_diag_pct), 4)
    
    print(
            f"Final diagonal Test Error= {test_diag_pct:.4f}% "
        )

    results[session]["test_error_at_best_validation"] = test_diag_pct





NORM_RANDOM Session: 01
csv_pfad: personalization/01/norm_labels.csv
Gesamte Daten: 498
Dataset Size: 498
Train:      348
Validation: 74
Test:       76

norm_random EXPERIMENT: 
	Dataset_type:         norm_labels.csv
	Dataset split:        norm_random
	Train size:           1000
	Test size:            200
	Batch size:           32
	Learning rate:        0.0001
	Activation:           Sigmoid
	Trainable layers:     FC only
	Max_Epochs:           500
	Patience:             3

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
		 Session: 01
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=

	 Initial evaluation: 


 Train_error:
MAE : 0.2571 	 RMSE: 0.2964 
Diagonal-Error %: 20.9575 %

 Validat_error:
MAE : 0.2376 	 RMSE: 0.2778 
Diagonal-Error %: 19.6418 %
Epoch 0 | Train error: 20.9575% | Validat error: 19.6418%


	 Training: 



Normalized | Epoch 1: 100%|██████████████████████████████████████████████████████████| 11/11 [00:15<00:00,  1.39s/it, loss=0.254]



 Train:
MAE : 0.2506 	 RMSE: 0.2889 
Diagonal-Error %: 20.4269 %

 Validat:
MAE : 0.2288 	 RMSE: 0.2693 
Diagonal-Error %: 19.0437 %

Validat-Diagonal-Error: 19.0437% | Improvement: 1

[13:49:52] Epoch 1: | 34.83 s | Loss: 0.2546 | Train-Error: 20.4269% | Test-Error: 19.0437%


Normalized | Epoch 2: 100%|██████████████████████████████████████████████████████████| 11/11 [00:18<00:00,  1.67s/it, loss=0.227]



 Train:
MAE : 0.2403 	 RMSE: 0.2781 
Diagonal-Error %: 19.6670 %

 Validat:
MAE : 0.2197 	 RMSE: 0.2586 
Diagonal-Error %: 18.2882 %

Validat-Diagonal-Error: 18.2882% | Improvement: 2

[13:50:26] Epoch 2: | 33.67 s | Loss: 0.2474 | Train-Error: 19.6670% | Test-Error: 18.2882%


...


 Train:
MAE : 0.2814 	 RMSE: 0.3243 
Diagonal-Error %: 22.9292 %

 Validat:
MAE : 0.3023 	 RMSE: 0.3456 
Diagonal-Error %: 24.4368 %

Patience: 1/3

[16:59:32] Epoch 6: | 75.81 s | Loss: 0.2870 | Train-Error: 22.9292% | Test-Error: 24.4368%


Normalized | Epoch 7: 100%|██████████████████████████████████████████████████████████| 25/25 [00:36<00:00,  1.45s/it, loss=0.288]



 Train:
MAE : 0.2874 	 RMSE: 0.3475 
Diagonal-Error %: 24.5755 %

 Validat:
MAE : 0.3189 	 RMSE: 0.3778 
Diagonal-Error %: 26.7162 %

Patience: 2/3

[17:00:48] Epoch 7: | 76.33 s | Loss: 0.2753 | Train-Error: 24.5754% | Test-Error: 26.7162%


Normalized | Epoch 8: 100%|██████████████████████████████████████████████████████████| 25/25 [00:37<00:00,  1.48s/it, loss=0.355]



 Train:
MAE : 0.2802 	 RMSE: 0.3449 
Diagonal-Error %: 24.3865 %

 Validat:
MAE : 0.3121 	 RMSE: 0.3754 
Diagonal-Error %: 26.5451 %

Patience: 3/3

Early Stopping after 8 epochs.
Imporovments totally: 4

totll running-tiems (s): 665.94 s
totll running-tiems (min): 11.10 min


NORM_RANDOM Session: 18 RESULTS: 
Best Validation Error:   24.2063%
Running Time:            11.10 min
Epochs:                  8
Best Epoch:              5
Improvements:            4
Test:

MAE : 0.3010 	 RMSE: 0.3394 
Diagonal-Error %: 23.9968 %
Final diagonal Test Error= 23.9968% 


In [37]:
for session, values in results.items():

    #  print(
    # f" Sessions: {session} |"
    # f" epochs: {values['epochs']} Epochen |"
    # f" running_time: {values['time_minutes']:.2f} min |"
    # f" improvements: {values['improvements']} Epochen |"
    # f" best_test_error: {values['best_test_error']:.4f} % |"
    # )

    print("\n" + "=" * 40)
    print(f"{def_dataset.upper()} Session: {session.upper()} RESULTS: ")
    print("=" * 40)
    
    print(
        f"Best Validation Error:                  {results[session]['best_validat_error']:.4f}%"
    )
    print(
        f"Tetst-Error at best validation:         {results[session]['test_error_at_best_validation']:.4f}%"
    )
    print(
        f"Running Time:                           {results[session]['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                                 {results[session]['epochs']}"
    )
    print(
        f"Best Epoch:                             {results[session]['best_epoch']}"
    )
    print(
        f"Improvements:                           {results[session]['improvements']}"
    )
    print("=" * 30)
    print("\n")


NORM_RANDOM Session: 01 RESULTS: 
Best Validation Error:   9.1266%
Best Test Error:         10.2422%
Running Time:            25.67 min
Epochs:                  47
Best Epoch:              44
Improvements:            29



NORM_RANDOM Session: 02 RESULTS: 
Best Validation Error:   9.4785%
Best Test Error:         9.1323%
Running Time:            7.82 min
Epochs:                  15
Best Epoch:              12
Improvements:            11



NORM_RANDOM Session: 03 RESULTS: 
Best Validation Error:   14.5845%
Best Test Error:         15.5043%
Running Time:            6.86 min
Epochs:                  12
Best Epoch:              9
Improvements:            7



NORM_RANDOM Session: 04 RESULTS: 
Best Validation Error:   12.4310%
Best Test Error:         11.1467%
Running Time:            8.64 min
Epochs:                  13
Best Epoch:              10
Improvements:            9



NORM_RANDOM Session: 05 RESULTS: 
Best Validation Error:   18.5911%
Best Test Error:         15.5761%
Running Ti

In [ ]:
# NORM_SUBJECT

========================================
NORM_SUBJECT Session: 01 RESULTS: 
========================================
Best Validation Error:   10.7186%
Best Test Error:         11.4291%
Running Time:            13.35 min
Epochs:                  20
Best Epoch:              17
Improvements:            14
==============================



========================================
NORM_SUBJECT Session: 02 RESULTS: 
========================================
Best Validation Error:   9.1110%
Best Test Error:         8.5165%
Running Time:            10.28 min
Epochs:                  16
Best Epoch:              13
Improvements:            12
==============================



========================================
NORM_SUBJECT Session: 03 RESULTS: 
========================================
Best Validation Error:   11.6067%
Best Test Error:         12.1000%
Running Time:            14.92 min
Epochs:                  25
Best Epoch:              22
Improvements:            17
==============================



========================================
NORM_SUBJECT Session: 04 RESULTS: 
========================================
Best Validation Error:   12.4003%
Best Test Error:         10.8391%
Running Time:            10.44 min
Epochs:                  16
Best Epoch:              13
Improvements:            11
==============================



========================================
NORM_SUBJECT Session: 05 RESULTS: 
========================================
Best Validation Error:   17.1120%
Best Test Error:         12.7782%
Running Time:            7.52 min
Epochs:                  30
Best Epoch:              27
Improvements:            22
==============================



========================================
NORM_SUBJECT Session: 06 RESULTS: 
========================================
Best Validation Error:   13.1890%
Best Test Error:         10.4886%
Running Time:            8.97 min
Epochs:                  30
Best Epoch:              27
Improvements:            23
==============================



========================================
NORM_SUBJECT Session: 07 RESULTS: 
========================================
Best Validation Error:   17.4645%
Best Test Error:         15.4393%
Running Time:            5.52 min
Epochs:                  25
Best Epoch:              22
Improvements:            18
==============================



========================================
NORM_SUBJECT Session: 08 RESULTS: 
========================================
Best Validation Error:   18.7970%
Best Test Error:         19.5001%
Running Time:            1.02 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 09 RESULTS: 
========================================
Best Validation Error:   19.3284%
Best Test Error:         18.5149%
Running Time:            1.85 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_SUBJECT Session: 10 RESULTS: 
========================================
Best Validation Error:   20.9562%
Best Test Error:         21.4255%
Running Time:            13.20 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_SUBJECT Session: 11 RESULTS: 
========================================
Best Validation Error:   24.0390%
Best Test Error:         25.5719%
Running Time:            4.31 min
Epochs:                  5
Best Epoch:              2
Improvements:            2
==============================



========================================
NORM_SUBJECT Session: 12 RESULTS: 
========================================
Best Validation Error:   21.0522%
Best Test Error:         20.4153%
Running Time:            39.41 min
Epochs:                  19
Best Epoch:              16
Improvements:            12
==============================



========================================
NORM_SUBJECT Session: 13 RESULTS: 
========================================
Best Validation Error:   20.3144%
Best Test Error:         20.1779%
Running Time:            11.56 min
Epochs:                  10
Best Epoch:              7
Improvements:            4
==============================



========================================
NORM_SUBJECT Session: 14 RESULTS: 
========================================
Best Validation Error:   22.5814%
Best Test Error:         22.4166%
Running Time:            3.12 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 15 RESULTS: 
========================================
Best Validation Error:   19.8634%
Best Test Error:         20.6428%
Running Time:            16.59 min
Epochs:                  15
Best Epoch:              12
Improvements:            8
==============================



========================================
NORM_SUBJECT Session: 16 RESULTS: 
========================================
Best Validation Error:   22.1187%
Best Test Error:         23.7873%
Running Time:            4.95 min
Epochs:                  4
Best Epoch:              1
Improvements:            1
==============================



========================================
NORM_SUBJECT Session: 17 RESULTS: 
========================================
Best Validation Error:   23.6528%
Best Test Error:         23.7854%
Running Time:            4.59 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 18 RESULTS: 
========================================
Best Validation Error:   23.3862%
Best Test Error:         23.8104%
Running Time:            12.42 min
Epochs:                  9
Best Epoch:              6
Improvements:            5
==============================

In [ ]:
# NORM_RANDOM

========================================
NORM_RANDOM Session: 01 RESULTS: 
========================================
Best Validation Error:   9.1266%
Best Test Error:         10.2422%
Running Time:            25.67 min
Epochs:                  47
Best Epoch:              44
Improvements:            29
==============================



========================================
NORM_RANDOM Session: 02 RESULTS: 
========================================
Best Validation Error:   9.4785%
Best Test Error:         9.1323%
Running Time:            7.82 min
Epochs:                  15
Best Epoch:              12
Improvements:            11
==============================



========================================
NORM_RANDOM Session: 03 RESULTS: 
========================================
Best Validation Error:   14.5845%
Best Test Error:         15.5043%
Running Time:            6.86 min
Epochs:                  12
Best Epoch:              9
Improvements:            7
==============================



========================================
NORM_RANDOM Session: 04 RESULTS: 
========================================
Best Validation Error:   12.4310%
Best Test Error:         11.1467%
Running Time:            8.64 min
Epochs:                  13
Best Epoch:              10
Improvements:            9
==============================



========================================
NORM_RANDOM Session: 05 RESULTS: 
========================================
Best Validation Error:   18.5911%
Best Test Error:         15.5761%
Running Time:            3.90 min
Epochs:                  15
Best Epoch:              12
Improvements:            12
==============================



========================================
NORM_RANDOM Session: 06 RESULTS: 
========================================
Best Validation Error:   13.5419%
Best Test Error:         11.0817%
Running Time:            6.40 min
Epochs:                  25
Best Epoch:              22
Improvements:            20
==============================



========================================
NORM_RANDOM Session: 07 RESULTS: 
========================================
Best Validation Error:   18.5788%
Best Test Error:         16.2649%
Running Time:            3.41 min
Epochs:                  19
Best Epoch:              16
Improvements:            16
==============================



========================================
NORM_RANDOM Session: 08 RESULTS: 
========================================
Best Validation Error:   18.6294%
Best Test Error:         20.1716%
Running Time:            1.01 min
Epochs:                  4
Best Epoch:              1
Improvements:            1
==============================



========================================
NORM_RANDOM Session: 09 RESULTS: 
========================================
Best Validation Error:   16.9268%
Best Test Error:         14.9900%
Running Time:            3.48 min
Epochs:                  17
Best Epoch:              14
Improvements:            12
==============================



========================================
NORM_RANDOM Session: 10 RESULTS: 
========================================
Best Validation Error:   21.0107%
Best Test Error:         20.6791%
Running Time:            13.72 min
Epochs:                  8
Best Epoch:              5
Improvements:            4
==============================



========================================
NORM_RANDOM Session: 11 RESULTS: 
========================================
Best Validation Error:   18.8287%
Best Test Error:         21.6370%
Running Time:            11.89 min
Epochs:                  18
Best Epoch:              15
Improvements:            11
==============================



========================================
NORM_RANDOM Session: 12 RESULTS: 
========================================
Best Validation Error:   21.0497%
Best Test Error:         20.6046%
Running Time:            29.15 min
Epochs:                  15
Best Epoch:              12
Improvements:            9
==============================



========================================
NORM_RANDOM Session: 13 RESULTS: 
========================================
Best Validation Error:   20.2498%
Best Test Error:         19.8629%
Running Time:            10.08 min
Epochs:                  9
Best Epoch:              6
Improvements:            5
==============================



========================================
NORM_RANDOM Session: 14 RESULTS: 
========================================
Best Validation Error:   22.6881%
Best Test Error:         22.3185%
Running Time:            3.10 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_RANDOM Session: 15 RESULTS: 
========================================
Best Validation Error:   18.8891%
Best Test Error:         19.7295%
Running Time:            27.15 min
Epochs:                  25
Best Epoch:              22
Improvements:            16
==============================



========================================
NORM_RANDOM Session: 16 RESULTS: 
========================================
Best Validation Error:   22.2695%
Best Test Error:         23.9960%
Running Time:            4.90 min
Epochs:                  4
Best Epoch:              1
Improvements:            1
==============================



========================================
NORM_RANDOM Session: 17 RESULTS: 
========================================
Best Validation Error:   21.2905%
Best Test Error:         21.2379%
Running Time:            13.35 min
Epochs:                  10
Best Epoch:              7
Improvements:            4
==============================



========================================
NORM_RANDOM Session: 18 RESULTS: 
========================================
Best Validation Error:   24.2063%
Best Test Error:         23.9968%
Running Time:            11.10 min
Epochs:                  8
Best Epoch:              5
Improvements:            4
==============================


## Norm-Subject:

In [40]:

name = f"{def_dataset}_results"

globals()[name] = results.copy()


for session, values in globals()[name].items():

    print("\n" + "=" * 40)
    print(f"{def_dataset.upper()} Session: {session.upper()}: ")
    print("=" * 40)
    
    print(
        f"Best Validation Error:                   {values['best_validat_error']:.4f}%"
    )
    print(
        f"Tetst-Error at best validation:          {values['test_error_at_best_validation']:.4f}%"
    )
    print(
        f"Running Time:                            {values['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                                  {values['epochs']}"
    )
    print(
        f"Best Epoch:                              {values['best_epoch']}"
    )
    print(
        f"Improvements:                            {values['improvements']}"
    )
    print("=" * 30)
    print("\n")





NORM_SUBJECT Session: 01 RESULTS: 
Best Validation Error:   10.7186%
Best Test Error:         11.4291%
Running Time:            13.35 min
Epochs:                  20
Best Epoch:              17
Improvements:            14



NORM_SUBJECT Session: 02 RESULTS: 
Best Validation Error:   9.1110%
Best Test Error:         8.5165%
Running Time:            10.28 min
Epochs:                  16
Best Epoch:              13
Improvements:            12



NORM_SUBJECT Session: 03 RESULTS: 
Best Validation Error:   11.6067%
Best Test Error:         12.1000%
Running Time:            14.92 min
Epochs:                  25
Best Epoch:              22
Improvements:            17



NORM_SUBJECT Session: 04 RESULTS: 
Best Validation Error:   12.4003%
Best Test Error:         10.8391%
Running Time:            10.44 min
Epochs:                  16
Best Epoch:              13
Improvements:            11



NORM_SUBJECT Session: 05 RESULTS: 
Best Validation Error:   17.1120%
Best Test Error:         12.7782

In [ ]:
========================================
NORM_SUBJECT Session: 01 RESULTS: 
========================================
Best Validation Error:   10.7186%
Best Test Error:         11.4291%
Running Time:            13.35 min
Epochs:                  20
Best Epoch:              17
Improvements:            14
==============================



========================================
NORM_SUBJECT Session: 02 RESULTS: 
========================================
Best Validation Error:   9.1110%
Best Test Error:         8.5165%
Running Time:            10.28 min
Epochs:                  16
Best Epoch:              13
Improvements:            12
==============================



========================================
NORM_SUBJECT Session: 03 RESULTS: 
========================================
Best Validation Error:   11.6067%
Best Test Error:         12.1000%
Running Time:            14.92 min
Epochs:                  25
Best Epoch:              22
Improvements:            17
==============================



========================================
NORM_SUBJECT Session: 04 RESULTS: 
========================================
Best Validation Error:   12.4003%
Best Test Error:         10.8391%
Running Time:            10.44 min
Epochs:                  16
Best Epoch:              13
Improvements:            11
==============================



========================================
NORM_SUBJECT Session: 05 RESULTS: 
========================================
Best Validation Error:   17.1120%
Best Test Error:         12.7782%
Running Time:            7.52 min
Epochs:                  30
Best Epoch:              27
Improvements:            22
==============================



========================================
NORM_SUBJECT Session: 06 RESULTS: 
========================================
Best Validation Error:   13.1890%
Best Test Error:         10.4886%
Running Time:            8.97 min
Epochs:                  30
Best Epoch:              27
Improvements:            23
==============================



========================================
NORM_SUBJECT Session: 07 RESULTS: 
========================================
Best Validation Error:   17.4645%
Best Test Error:         15.4393%
Running Time:            5.52 min
Epochs:                  25
Best Epoch:              22
Improvements:            18
==============================



========================================
NORM_SUBJECT Session: 08 RESULTS: 
========================================
Best Validation Error:   18.7970%
Best Test Error:         19.5001%
Running Time:            1.02 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 09 RESULTS: 
========================================
Best Validation Error:   19.3284%
Best Test Error:         18.5149%
Running Time:            1.85 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_SUBJECT Session: 10 RESULTS: 
========================================
Best Validation Error:   20.9562%
Best Test Error:         21.4255%
Running Time:            13.20 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_SUBJECT Session: 11 RESULTS: 
========================================
Best Validation Error:   24.0390%
Best Test Error:         25.5719%
Running Time:            4.31 min
Epochs:                  5
Best Epoch:              2
Improvements:            2
==============================



========================================
NORM_SUBJECT Session: 12 RESULTS: 
========================================
Best Validation Error:   21.0522%
Best Test Error:         20.4153%
Running Time:            39.41 min
Epochs:                  19
Best Epoch:              16
Improvements:            12
==============================



========================================
NORM_SUBJECT Session: 13 RESULTS: 
========================================
Best Validation Error:   20.3144%
Best Test Error:         20.1779%
Running Time:            11.56 min
Epochs:                  10
Best Epoch:              7
Improvements:            4
==============================



========================================
NORM_SUBJECT Session: 14 RESULTS: 
========================================
Best Validation Error:   22.5814%
Best Test Error:         22.4166%
Running Time:            3.12 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 15 RESULTS: 
========================================
Best Validation Error:   19.8634%
Best Test Error:         20.6428%
Running Time:            16.59 min
Epochs:                  15
Best Epoch:              12
Improvements:            8
==============================



========================================
NORM_SUBJECT Session: 16 RESULTS: 
========================================
Best Validation Error:   22.1187%
Best Test Error:         23.7873%
Running Time:            4.95 min
Epochs:                  4
Best Epoch:              1
Improvements:            1
==============================



========================================
NORM_SUBJECT Session: 17 RESULTS: 
========================================
Best Validation Error:   23.6528%
Best Test Error:         23.7854%
Running Time:            4.59 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_SUBJECT Session: 18 RESULTS: 
========================================
Best Validation Error:   23.3862%
Best Test Error:         23.8104%
Running Time:            12.42 min
Epochs:                  9
Best Epoch:              6
Improvements:            5
==============================

### Norm-Random:

In [38]:
name = f"{def_dataset}_results"
globals()[name] = results.copy()


for session, values in globals()[name].items():

    print("\n" + "=" * 40)
    print(f"{def_dataset.upper()} Session: {session.upper()}: ")
    print("=" * 40)
    
    print(
        f"Best Validation Error:                   {values['best_validat_error']:.4f}%"
    )
    print(
        f"Tetst-Error at best validation:          {values['test_error_at_best_validation']:.4f}%"
    )
    print(
        f"Running Time:                            {values['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                                  {values['epochs']}"
    )
    print(
        f"Best Epoch:                              {values['best_epoch']}"
    )
    print(
        f"Improvements:                            {values['improvements']}"
    )
    print("=" * 30)
    print("\n")




NORM_RANDOM Session: 01: 
Best Validation Error:   10.7186%
Best Test Error:         11.4291%
Running Time:            13.35 min
Epochs:                  20
Best Epoch:              17
Improvements:            14



NORM_RANDOM Session: 02: 
Best Validation Error:   9.1110%
Best Test Error:         8.5165%
Running Time:            10.28 min
Epochs:                  16
Best Epoch:              13
Improvements:            12



NORM_RANDOM Session: 03: 
Best Validation Error:   11.6067%
Best Test Error:         12.1000%
Running Time:            14.92 min
Epochs:                  25
Best Epoch:              22
Improvements:            17



NORM_RANDOM Session: 04: 
Best Validation Error:   12.4003%
Best Test Error:         10.8391%
Running Time:            10.44 min
Epochs:                  16
Best Epoch:              13
Improvements:            11



NORM_RANDOM Session: 05: 
Best Validation Error:   17.1120%
Best Test Error:         12.7782%
Running Time:            7.52 min
Epochs:  

In [ ]:
========================================
NORM_RANDOM Session: 01: 
========================================
Best Validation Error:   10.7186%
Best Test Error:         11.4291%
Running Time:            13.35 min
Epochs:                  20
Best Epoch:              17
Improvements:            14
==============================



========================================
NORM_RANDOM Session: 02: 
========================================
Best Validation Error:   9.1110%
Best Test Error:         8.5165%
Running Time:            10.28 min
Epochs:                  16
Best Epoch:              13
Improvements:            12
==============================



========================================
NORM_RANDOM Session: 03: 
========================================
Best Validation Error:   11.6067%
Best Test Error:         12.1000%
Running Time:            14.92 min
Epochs:                  25
Best Epoch:              22
Improvements:            17
==============================



========================================
NORM_RANDOM Session: 04: 
========================================
Best Validation Error:   12.4003%
Best Test Error:         10.8391%
Running Time:            10.44 min
Epochs:                  16
Best Epoch:              13
Improvements:            11
==============================



========================================
NORM_RANDOM Session: 05: 
========================================
Best Validation Error:   17.1120%
Best Test Error:         12.7782%
Running Time:            7.52 min
Epochs:                  30
Best Epoch:              27
Improvements:            22
==============================



========================================
NORM_RANDOM Session: 06: 
========================================
Best Validation Error:   13.1890%
Best Test Error:         10.4886%
Running Time:            8.97 min
Epochs:                  30
Best Epoch:              27
Improvements:            23
==============================



========================================
NORM_RANDOM Session: 07: 
========================================
Best Validation Error:   17.4645%
Best Test Error:         15.4393%
Running Time:            5.52 min
Epochs:                  25
Best Epoch:              22
Improvements:            18
==============================



========================================
NORM_RANDOM Session: 08: 
========================================
Best Validation Error:   18.7970%
Best Test Error:         19.5001%
Running Time:            1.02 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_RANDOM Session: 09: 
========================================
Best Validation Error:   19.3284%
Best Test Error:         18.5149%
Running Time:            1.85 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_RANDOM Session: 10: 
========================================
Best Validation Error:   20.9562%
Best Test Error:         21.4255%
Running Time:            13.20 min
Epochs:                  6
Best Epoch:              3
Improvements:            3
==============================



========================================
NORM_RANDOM Session: 11: 
========================================
Best Validation Error:   24.0390%
Best Test Error:         25.5719%
Running Time:            4.31 min
Epochs:                  5
Best Epoch:              2
Improvements:            2
==============================



========================================
NORM_RANDOM Session: 12: 
========================================
Best Validation Error:   21.0522%
Best Test Error:         20.4153%
Running Time:            39.41 min
Epochs:                  19
Best Epoch:              16
Improvements:            12
==============================



========================================
NORM_RANDOM Session: 13: 
========================================
Best Validation Error:   20.3144%
Best Test Error:         20.1779%
Running Time:            11.56 min
Epochs:                  10
Best Epoch:              7
Improvements:            4
==============================



========================================
NORM_RANDOM Session: 14: 
========================================
Best Validation Error:   22.5814%
Best Test Error:         22.4166%
Running Time:            3.12 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_RANDOM Session: 15: 
========================================
Best Validation Error:   19.8634%
Best Test Error:         20.6428%
Running Time:            16.59 min
Epochs:                  15
Best Epoch:              12
Improvements:            8
==============================



========================================
NORM_RANDOM Session: 16: 
========================================
Best Validation Error:   22.1187%
Best Test Error:         23.7873%
Running Time:            4.95 min
Epochs:                  4
Best Epoch:              1
Improvements:            1
==============================



========================================
NORM_RANDOM Session: 17: 
========================================
Best Validation Error:   23.6528%
Best Test Error:         23.7854%
Running Time:            4.59 min
Epochs:                  3
Best Epoch:              0
Improvements:            0
==============================



========================================
NORM_RANDOM Session: 18: 
========================================
Best Validation Error:   23.3862%
Best Test Error:         23.8104%
Running Time:            12.42 min
Epochs:                  9
Best Epoch:              6
Improvements:            5
==============================

In [22]:
sess1 = ['10', '11','12', '13', '14', '15', '16', '17', '18']

sess2 = ['19', '20', '21', '22', '23', '24', '25', '26', '27']

sess  = sess1 + sess2
print(sess)

['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27']


In [18]:
patience = 3
print(f"\t Patience:             {patience}")

#============================================================================
#                 normalize_Subject
#============================================================================
# def_dataset = dataset_type[0]                             # norm_subject

#============================================================================
#                 normalize_random
#============================================================================
# def_dataset = dataset_type[1]                             # norm_random


# best_saved_model = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"
print(f"\n model_name: \t\t {best_saved_model}")





test_results = {}

sess1 = ['10', '11','12', '13', '14', '15', '16', '17', '18']
sess2 = ['19', '20', '21', '22', '23', '24', '25', '26', '27']

sess = sess1 + sess2


# sess = ['17','18']
# sess = ['26','27']

for s in sess: 
# for s in sessions:

    session = s

    print("\n" + "=" * 70)
    print(f"{def_dataset.upper()} Session: {session.upper()}")
    print("=" * 70)
    
    dirs = f"personalization/{session}"
    
    csv_path = Path(dirs) / "norm_labels_clean.csv"
    
    if not csv_path.exists():
        raise FileNotFoundError(f"Fehlt: {csv_path.resolve()}")
    
    print(f"csv_pfad: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    dataset_size = len(df)
    
    print(f"Gesamte Daten: {dataset_size}")
    
    
    # ============================================================
    # Full Dataset
    # ============================================================
    
    # full_dataset = PersonalGazeDataset(
    #     root_dir=dirs,
    #     transform=None,
    # )

    full_dataset = PersonalGazeDataset(
        root_dir=dirs,
        dataset_name="norm_labels_clean.csv"
    )

    dataset_size = len(full_dataset)

    print(f"Dataset Size: {dataset_size}")
    

    # ============================================================
    # Reproduzierbare Splits-Indizes
    # ============================================================

        
    generator = torch.Generator().manual_seed(42)
    
    indices = torch.randperm(
        dataset_size,
        generator=generator
    ).tolist()
    
    # ============================================================
    # Split Sizes
    # ============================================================
    
    train_size = int(dataset_size * 0.70)
    validat_size = int(dataset_size * 0.15)
    
    test_size = (
        dataset_size
        - train_size
        - validat_size
    )
    
    train_indices = indices[:train_size]
    
    validat_indices = indices[
        train_size:train_size + validat_size
    ]
    
    test_indices = indices[
        train_size + validat_size:
    ]
    

    # ============================================================
    # 2. Subsets mit getrennten Transformationen
    # ============================================================

    train_dataset = TransformSubset(
        full_dataset,
        train_indices,
        transform=train_transform
    )

    validat_dataset = TransformSubset(
        full_dataset,
        validat_indices,
        transform=eval_transform
    )

    test_dataset = TransformSubset(
        full_dataset,
        test_indices,
        transform=eval_transform
    )


    # ============================================================
    # 3. Kontrolle der Dataset-Größen
    # ============================================================

    print(f"Train:      {len(train_dataset)}")
    print(f"Validation: {len(validat_dataset)}")
    print(f"Test:       {len(test_dataset)}")


    # ============================================================
    # 4. DataLoader
    # ============================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    validat_loader = DataLoader(
        validat_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )

##########################################################################
##########################################################################
    train_eval_dataset = TransformSubset(
        full_dataset,
        train_indices,
        transform=eval_transform
    )

    train_eval_loader = DataLoader(
        train_eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )



    # ============================================================
    # Training:
    # ============================================================
    
    # acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']
    act = "Sigmoid"
    
    trainable_layers = "FC only"
    
    
    print("\n" + "=" * 70)
    print(f"{def_dataset} EXPERIMENT: ")
    
     
    print(f"\t Dataset Type:         {dataset_name}")
    print(f"\t Dataset Split:        {def_dataset}")
    print(f"\t Train Size:           {def_dataset_size[0]}")
    print(f"\t Test Size:            {def_dataset_size[1]}")
    print(f"\t Batch Size:           {batch_size}")
    print(f"\t Learning Rate:        {learning_rate}")
    print(f"\t Activation:           {act}")
    print(f"\t Trainable Layers:     {trainable_layers}")
    print(f"\t Loaded_Model:         {best_saved_model}")
    print(f"\t Max_Epochs:           {epochs}")
    print(f"\t Patience:             {patience}")
    print("=" * 70)
    
    
    
    
    if globals().get("bas_const_err") is None:
        bas_const_err = 27.245
    
    if globals().get("bas_rand_err") is None:
        bas_rand_err = 38.961
    
    
    
    # --------------------------------------------------------
    # Start total training timer
    # --------------------------------------------------------
    
    train_start = time.perf_counter()
    
    
    
    epochs = epochs
    
    
    
    model_output = './models/best_models/personal/test/'
    model_dir= Path(model_output)
    
    if not os.path.exists(model_dir):
        model_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    diagrams_output = './results/best_diagrams/personal'
    diagrams_dir= Path(diagrams_output)
    
    if not os.path.exists(diagrams_dir):
        diagrams_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    
    # ============================================================
    # Store train and test error curves
    # ============================================================
    
    # train_errors = []
    # test_errors = []
    
    best_error = float("inf")
    
    patience_counter = 0                        # Number Epochen without Optimierung
    improvements = 0                            # Number Epochen with Optimierung
    best_epoch = 0
    
    diag_train_errors = []
    diag_validat_errors  = []
    
    act_time_epochs = {}
    
    
    
    criterion = nn.L1Loss()
    
    criterion_base = nn.L1Loss()
    
    # --------------------------------------------------------
    # Reset model
    # --------------------------------------------------------
    
    model, model_name = reset_model(act=act)

    checkpoint = torch.load(
        # checkpoint_path,
        best_saved_model,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    # --------------------------------------------------------
    # Freeze all parameters
    # --------------------------------------------------------
    
    for param in model.parameters():
        param.requires_grad = False
    
    
    # --------------------------------------------------------
    # Unfreeze ONLY FC
    # --------------------------------------------------------
    
    for param in model.layer4.parameters():
        param.requires_grad=False
    
    
    for param in model.fc.parameters():
        param.requires_grad = True
    
    
    
    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_Decay
    )
    
    
    
    
    
    print('\n ',"=#=" * 25)
    print(f"\t\t Session: {session}")
    print(' ',"=#=" * 25)
    
    
    # ============================================================
    # Initial evaluation before training
    # ============================================================
    print("\n\t Initial evaluation: \n")
    
    model.eval()
    
    print("\n Train_error:")
    train_mae, train_rmse, train_diag_pct = diagonal_errors(
        model,
        train_loader,
        device
    )

    # train_mae, train_rmse, train_diag_pct = diagonal_errors(
    #     model,
    #     train_eval_loader,
    #     device
    # )
    
    train_diag_pct = np.round(float(train_diag_pct), 4)
    # diag_train_errors.append(train_diag_pct)
    
    
    
    
    print("\n Validat_error:")
    validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
        model,
        validat_loader,
        device
    )
    
    validat_diag_pct = np.round(float(validat_diag_pct), 4)
    # diag_validat_errors.append(validat_diag_pct)
    
    
    
    print(
        f"Epoch 0 | "
        f"Train error: {train_diag_pct:.4f}% | "
        f"Validat error: {validat_diag_pct:.4f}%"
    )
    
    
    
    # Initial best test error
    if validat_diag_pct < best_error:  
        best_error = validat_diag_pct
        best_epoch = 0
    
    
    # save the better Modell
    torch.save(
        model.state_dict(),
        f"{model_output}"
        f"best_{model_name}_{act}_{def_dataset}_{session}_test.path"
    )
    
    # ============================================================
    # Training
    # ============================================================
    
    print("\n\n\t Training: \n")
    
    for epoch in range(epochs):
    
        epoch_start = time.perf_counter()
    
        model.train()
    
        running_loss = 0.0
    
    
        loop = tqdm(
            train_loader,
            desc=f"Normalized | Epoch {epoch + 1}"
        )
        
    
        for images, targets in loop:
            
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            preds = model(images)
    
            ################################
            # Loss 1:
            ################################
            
            # loss = criterion(
            #     preds,
            #     targets
            # )
    
    
            
            ################################
            # Loss 2: 
            ################################
            
            loss = sensitive_loss(
                preds,
                targets
            )
            
            ################################
    
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
            loop.set_postfix(
                loss=loss.item()
            )
    
        
    
        # ========================================================
        # Evaluation after epoch
        # ========================================================
    
        model.eval()
    
    
        # --------------------------------------------------------
        # Train error
        # --------------------------------------------------------
        print("\n Train:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(
            model,
            train_loader,
            device
        )

        # train_mae, train_rmse, train_diag_pct = diagonal_errors(
        #     model,
        #     train_eval_loader,
        #     device
        # ) 
   
        
    
        train_diag_pct = np.round(float(train_diag_pct), 4)
        diag_train_errors.append(train_diag_pct)
    
    
        # --------------------------------------------------------
        # Test error
        # --------------------------------------------------------
        print("\n Validat:")
        validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
            model,
            validat_loader,
            device
        )
    
        validat_diag_pct = np.round(float(validat_diag_pct), 4)
        diag_validat_errors.append(validat_diag_pct)
    
    
        # --------------------------------------------------
        # Early Stopping
        # --------------------------------------------------
    
        if validat_diag_pct < best_error:
    
            best_error = validat_diag_pct
    
            best_epoch = epoch + 1
    
            patience_counter = 0
            improvements += 1
    
            print(
                f"\nValidat-Diagonal-Error: "
                f"{validat_diag_pct:.4f}% | "
                f"Improvement: {improvements}"
            )
    
            # save the better Modell
            torch.save(
                model.state_dict(),
                f"{model_output}"
                f"best_{model_name}_{act}_{def_dataset}_{session}_test.path"
            )
        
    
        else:
    
            # without imporovement
            patience_counter += 1
    
            print(
                f"\nPatience: "
                f"{patience_counter}/{patience}"
            )
    
    
            if patience_counter >= patience:
    
                print(
                    f"\nEarly Stopping after "
                    f"{epoch + 1} epochs."
                )
                print(
                    f"Imporovments totally: {improvements}"
                )
    
                break
        
    
        # ========================================================
        # Epoch information
        # ========================================================
    
        epoch_end = time.perf_counter()
    
        epoch_time = epoch_end - epoch_start
    
    
        print(
            f"\n[{datetime.now().strftime('%H:%M:%S')}] "
            f"Epoch {epoch + 1}: | " 
            f"{epoch_time:.2f} s | "
            f"Loss: {running_loss / len(train_loader):.4f} | "
            f"Train-Error: {train_diag_pct:.4f}% | "
            f"Test-Error: {validat_diag_pct:.4f}%"
        )
    
        torch.save(
            model.state_dict(),
            f"{model_output}"
            f"last_{model_name}_{act}_{def_dataset}_{session}_test.path"
        )
    
    
    
    # ============================================================
    # Total running time
    # ============================================================
    
    train_end = time.perf_counter()
    
    elapsed_running_time = train_end - train_start
    
    elapsed_minutes = elapsed_running_time / 60
    
    print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} s")
    print(f"totll running-tiems (min): {elapsed_minutes:.2f} min\n")
    
    # ============================================================
    # Number of completed epochs
    # ============================================================
    
    epochs_completed = len(diag_validat_errors)
    
    
    # # ============================================================
    # # Store normalized-dataset results
    # # ============================================================
    
    test_results[session] = {
        "split_type" : def_dataset,
        "train_errors": diag_train_errors,
        "validat_errors": diag_validat_errors,
        "best_validat_error": best_error,
        "running_time_minutes": elapsed_minutes,
        "epochs": epochs_completed,
        "best_epoch": best_epoch,
        "improvements": improvements
    }
    
    
    # ========================================================
    # Print result for this learning rate
    # ========================================================
    
    print("\n" + "=" * 70)
    print(f"{def_dataset.upper()} Session: {session.upper()} RESULTS: ")
    print("=" * 70)
    
    print(
        f"Best Validation Error:   {test_results[session]['best_validat_error']:.4f}%"
    )
    print(
        f"Running Time:            {test_results[session]['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                  {test_results[session]['epochs']}"
    )
    print(
        f"Best Epoch:              {test_results[session]['best_epoch']}"
    )
    print(
        f"Improvements:            {test_results[session]['improvements']}"
    )
    print("=" * 70)
    
    
    # ---------- Evaluation ----------
    
    # norm_subject:
    # saved_model1 = "./models/best_models/best_ResNet_Sigmoid_norm_subject.path"
    # saved_model1 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"
    
    # norm_random:
    # saved_model2 = "./models/best_models/best_ResNet_Sigmoid_norm_random.path"
    # saved_model2 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"

    test_saved_model = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}_test.path"

    checkpoint = torch.load(
        test_saved_model,
        map_location=device
    )
    
    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    model.eval()
    
    
    print(f"Test:\n")
    test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
    
    test_diag_pct = np.round(float(test_diag_pct), 4)
    
    print(
            f"Final diagonal Test Error= {test_diag_pct:.4f}% "
        )

    test_results[session]["test_error_at_best_validation"] = test_diag_pct




	 Patience:             3

 model_name: 		 ./models/best_models/best_ResNet_Sigmoid_norm_subject.path

NORM_SUBJECT Session: 10
csv_pfad: personalization/10/norm_labels_clean.csv
Gesamte Daten: 1468
Dataset Size: 1468
Train:      1027
Validation: 220
Test:       221

norm_subject EXPERIMENT: 
	 Dataset Type:         norm_labels_clean.csv
	 Dataset Split:        norm_subject
	 Train Size:           1000
	 Test Size:            200
	 Batch Size:           32
	 Learning Rate:        0.0001
	 Activation:           Sigmoid
	 Trainable Layers:     FC only
	 Loaded_Model:         ./models/best_models/best_ResNet_Sigmoid_norm_subject.path
	 Max_Epochs:           500
	 Patience:             3

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
		 Session: 10
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=

	 Initial evaluation: 


 Train_error:
MAE : 0.2905 	 RMSE: 0.3229 
Diagonal-Error %: 22.8349 %

 Validat_error:
MAE : 0.2866 	 RMSE: 

Normalized | Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:41<00:00,  1.26s/it, loss=0.375]



 Train:
MAE : 0.2830 	 RMSE: 0.3172 
Diagonal-Error %: 22.4317 %

 Validat:
MAE : 0.2823 	 RMSE: 0.3166 
Diagonal-Error %: 22.3841 %

Validat-Diagonal-Error: 22.3841% | Improvement: 1

[00:52:44] Epoch 1: | 87.86 s | Loss: 0.4742 | Train-Error: 22.4316% | Test-Error: 22.3841%


Normalized | Epoch 2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:43<00:00,  1.32s/it, loss=0.447]



 Train:
MAE : 0.2671 	 RMSE: 0.3054 
Diagonal-Error %: 21.5977 %

 Validat:
MAE : 0.2714 	 RMSE: 0.3101 
Diagonal-Error %: 21.9239 %

Validat-Diagonal-Error: 21.9239% | Improvement: 2

[00:54:15] Epoch 2: | 90.94 s | Loss: 0.4611 | Train-Error: 21.5977% | Test-Error: 21.9239%


...


 Train:
MAE : 0.2012 	 RMSE: 0.2587 
Diagonal-Error %: 18.2945 %

 Validat:
MAE : 0.2184 	 RMSE: 0.2698 
Diagonal-Error %: 19.0754 %

Patience: 1/3

[04:22:34] Epoch 11: | 37.11 s | Loss: 0.3419 | Train-Error: 18.2944% | Test-Error: 19.0754%


Normalized | Epoch 12: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:18<00:00,  1.53s/it, loss=0.342]



 Train:
MAE : 0.1920 	 RMSE: 0.2534 
Diagonal-Error %: 17.9187 %

 Validat:
MAE : 0.2217 	 RMSE: 0.2937 
Diagonal-Error %: 20.7712 %

Patience: 2/3

[04:23:13] Epoch 12: | 38.91 s | Loss: 0.3265 | Train-Error: 17.9187% | Test-Error: 20.7712%


Normalized | Epoch 13: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:17<00:00,  1.48s/it, loss=0.352]



 Train:
MAE : 0.1865 	 RMSE: 0.2570 
Diagonal-Error %: 18.1734 %

 Validat:
MAE : 0.2137 	 RMSE: 0.2895 
Diagonal-Error %: 20.4722 %

Patience: 3/3

Early Stopping after 13 epochs.
Imporovments totally: 8

totll running-tiems (s): 512.97 s
totll running-tiems (min): 8.55 min


NORM_SUBJECT Session: 27 RESULTS: 
Best Validation Error:   18.7631%
Running Time:            8.55 min
Epochs:                  13
Best Epoch:              10
Improvements:            8
Test:

MAE : 0.2013 	 RMSE: 0.2557 
Diagonal-Error %: 18.0842 %
Final diagonal Test Error= 18.0842% 


In [19]:
name = f"{def_dataset}_test_results"
globals()[name] = test_results.copy()


for session, values in globals()[name].items():

    print("\n" + "=" * 40)
    print(f"{def_dataset.upper()} Session: {session.upper()}: ")
    print("=" * 40)
    
    print(
        f"Best Validation Error:                   {values['best_validat_error']:.4f}%"
    )
    print(
        f"Tetst-Error at best validation:          {values['test_error_at_best_validation']:.4f}%"
    )
    print(
        f"Running Time:                            {values['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:                                  {values['epochs']}"
    )
    print(
        f"Best Epoch:                              {values['best_epoch']}"
    )
    print(
        f"Improvements:                            {values['improvements']}"
    )
    print("=" * 30)
    print("\n")



NORM_SUBJECT Session: 10: 
Best Validation Error:                   20.9274%
Tetst-Error at best validation:          21.1872%
Running Time:                            11.53 min
Epochs:                                  7
Best Epoch:                              4
Improvements:                            4



NORM_SUBJECT Session: 11: 
Best Validation Error:                   24.2208%
Tetst-Error at best validation:          26.2037%
Running Time:                            3.41 min
Epochs:                                  5
Best Epoch:                              2
Improvements:                            2



NORM_SUBJECT Session: 12: 
Best Validation Error:                   21.3445%
Tetst-Error at best validation:          20.9778%
Running Time:                            22.67 min
Epochs:                                  13
Best Epoch:                              10
Improvements:                            8



NORM_SUBJECT Session: 13: 
Best Validation Error:                   

In [ ]:
# Normal Code:

========================================
NORM_SUBJECT Session: 19: 
========================================
Best Validation Error:                   14.0122%
Tetst-Error at best validation:          12.4800%
Running Time:                            21.18 min
Epochs:                                  34
Best Epoch:                              31
Improvements:                            19
==============================



========================================
NORM_SUBJECT Session: 20: 
========================================
Best Validation Error:                   14.9563%
Tetst-Error at best validation:          16.1742%
Running Time:                            9.68 min
Epochs:                                  15
Best Epoch:                              12
Improvements:                            12
==============================



========================================
NORM_SUBJECT Session: 21: 
========================================
Best Validation Error:                   8.6580%
Tetst-Error at best validation:          9.3512%
Running Time:                            12.19 min
Epochs:                                  14
Best Epoch:                              11
Improvements:                            10
==============================



========================================
NORM_SUBJECT Session: 22: 
========================================
Best Validation Error:                   11.8545%
Tetst-Error at best validation:          11.8010%
Running Time:                            11.69 min
Epochs:                                  17
Best Epoch:                              14
Improvements:                            11
==============================



========================================
NORM_SUBJECT Session: 23: 
========================================
Best Validation Error:                   15.0827%
Tetst-Error at best validation:          15.1915%
Running Time:                            15.50 min
Epochs:                                  18
Best Epoch:                              15
Improvements:                            13
==============================



========================================
NORM_SUBJECT Session: 24: 
========================================
Best Validation Error:                   20.1407%
Tetst-Error at best validation:          16.4405%
Running Time:                            8.39 min
Epochs:                                  10
Best Epoch:                              7
Improvements:                            7
==============================



========================================
NORM_SUBJECT Session: 25: 
========================================
Best Validation Error:                   22.0912%
Tetst-Error at best validation:          19.1398%
Running Time:                            6.09 min
Epochs:                                  7
Best Epoch:                              4
Improvements:                            3
==============================



========================================
NORM_SUBJECT Session: 26: 
========================================
Best Validation Error:                   15.0829%
Tetst-Error at best validation:          13.8051%
Running Time:                            16.61 min
Epochs:                                  20
Best Epoch:                              17
Improvements:                            15
==============================



========================================
NORM_SUBJECT Session: 27: 
========================================
Best Validation Error:                   18.9188%
Tetst-Error at best validation:          18.6520%
Running Time:                            7.16 min
Epochs:                                  11
Best Epoch:                              8
Improvements:                            8
==============================


In [ ]:
# Normal Code:
========================================
NORM_SUBJECT Session: 26: 
========================================
Best Validation Error:                   14.9322%
Tetst-Error at best validation:          13.3481%
Running Time:                            16.19 min
Epochs:                                  21
Best Epoch:                              18
Improvements:                            13
==============================



========================================
NORM_SUBJECT Session: 27: 
========================================
Best Validation Error:                   19.5005%
Tetst-Error at best validation:          19.3828%
Running Time:                            7.13 min
Epochs:                                  11
Best Epoch:                              8
Improvements:                            8
==============================


## Clear Code:

In [ ]:
# clear Code:

========================================
NORM_SUBJECT Session: 26: 
========================================
Best Validation Error:                   15.1363%
Tetst-Error at best validation:          13.5950%
Running Time:                            14.36 min
Epochs:                                  18
Best Epoch:                              15
Improvements:                            13
==============================



========================================
NORM_SUBJECT Session: 27: 
========================================
Best Validation Error:                   18.6618%
Tetst-Error at best validation:          17.9640%
Running Time:                            9.12 min
Epochs:                                  14
Best Epoch:                              11
Improvements:                            9
==============================

In [ ]:
# Clear Code:

========================================
NORM_SUBJECT Session: 17: 
========================================
Best Validation Error:                   23.6528%
Tetst-Error at best validation:          23.7854%
Running Time:                            3.90 min
Epochs:                                  3
Best Epoch:                              0
Improvements:                            0
==============================



========================================
NORM_SUBJECT Session: 18: 
========================================
Best Validation Error:                   24.9106%
Tetst-Error at best validation:          25.4427%
Running Time:                            5.44 min
Epochs:                                  4
Best Epoch:                              1
Improvements:                            1
==============================

In [ ]:
========================================
NORM_SUBJECT Session: 10: 
========================================
Best Validation Error:                   20.9274%
Tetst-Error at best validation:          21.1872%
Running Time:                            11.53 min
Epochs:                                  7
Best Epoch:                              4
Improvements:                            4
==============================



========================================
NORM_SUBJECT Session: 11: 
========================================
Best Validation Error:                   24.2208%
Tetst-Error at best validation:          26.2037%
Running Time:                            3.41 min
Epochs:                                  5
Best Epoch:                              2
Improvements:                            2
==============================



========================================
NORM_SUBJECT Session: 12: 
========================================
Best Validation Error:                   21.3445%
Tetst-Error at best validation:          20.9778%
Running Time:                            22.67 min
Epochs:                                  13
Best Epoch:                              10
Improvements:                            8
==============================



========================================
NORM_SUBJECT Session: 13: 
========================================
Best Validation Error:                   20.3425%
Tetst-Error at best validation:          20.1829%
Running Time:                            10.01 min
Epochs:                                  9
Best Epoch:                              6
Improvements:                            5
==============================



========================================
NORM_SUBJECT Session: 14: 
========================================
Best Validation Error:                   22.5814%
Tetst-Error at best validation:          22.4166%
Running Time:                            2.96 min
Epochs:                                  3
Best Epoch:                              0
Improvements:                            0
==============================



========================================
NORM_SUBJECT Session: 15: 
========================================
Best Validation Error:                   20.1330%
Tetst-Error at best validation:          20.7841%
Running Time:                            11.30 min
Epochs:                                  10
Best Epoch:                              7
Improvements:                            6
==============================



========================================
NORM_SUBJECT Session: 16: 
========================================
Best Validation Error:                   22.1227%
Tetst-Error at best validation:          23.7597%
Running Time:                            4.87 min
Epochs:                                  4
Best Epoch:                              1
Improvements:                            1
==============================



========================================
NORM_SUBJECT Session: 17: 
========================================
Best Validation Error:                   23.6528%
Tetst-Error at best validation:          23.7854%
Running Time:                            4.54 min
Epochs:                                  3
Best Epoch:                              0
Improvements:                            0
==============================



========================================
NORM_SUBJECT Session: 18: 
========================================
Best Validation Error:                   23.6379%
Tetst-Error at best validation:          23.1504%
Running Time:                            16.72 min
Epochs:                                  12
Best Epoch:                              9
Improvements:                            5
==============================



========================================
NORM_SUBJECT Session: 19: 
========================================
Best Validation Error:                   13.5924%
Tetst-Error at best validation:          11.7885%
Running Time:                            27.87 min
Epochs:                                  41
Best Epoch:                              38
Improvements:                            24
==============================



========================================
NORM_SUBJECT Session: 20: 
========================================
Best Validation Error:                   13.0972%
Tetst-Error at best validation:          12.2980%
Running Time:                            13.67 min
Epochs:                                  21
Best Epoch:                              18
Improvements:                            14
==============================



========================================
NORM_SUBJECT Session: 21: 
========================================
Best Validation Error:                   8.4815%
Tetst-Error at best validation:          9.3477%
Running Time:                            12.98 min
Epochs:                                  15
Best Epoch:                              12
Improvements:                            11
==============================



========================================
NORM_SUBJECT Session: 22: 
========================================
Best Validation Error:                   11.9396%
Tetst-Error at best validation:          12.4755%
Running Time:                            10.93 min
Epochs:                                  17
Best Epoch:                              14
Improvements:                            11
==============================



========================================
NORM_SUBJECT Session: 23: 
========================================
Best Validation Error:                   17.0901%
Tetst-Error at best validation:          17.0680%
Running Time:                            7.14 min
Epochs:                                  8
Best Epoch:                              5
Improvements:                            5
==============================



========================================
NORM_SUBJECT Session: 24: 
========================================
Best Validation Error:                   20.1008%
Tetst-Error at best validation:          16.8279%
Running Time:                            7.94 min
Epochs:                                  9
Best Epoch:                              6
Improvements:                            6
==============================



========================================
NORM_SUBJECT Session: 25: 
========================================
Best Validation Error:                   18.8294%
Tetst-Error at best validation:          15.6494%
Running Time:                            19.86 min
Epochs:                                  24
Best Epoch:                              21
Improvements:                            12
==============================



========================================
NORM_SUBJECT Session: 26: 
========================================
Best Validation Error:                   14.9355%
Tetst-Error at best validation:          13.5404%
Running Time:                            14.63 min
Epochs:                                  17
Best Epoch:                              14
Improvements:                            11
==============================



========================================
NORM_SUBJECT Session: 27: 
========================================
Best Validation Error:                   18.7631%
Tetst-Error at best validation:          18.0842%
Running Time:                            8.55 min
Epochs:                                  13
Best Epoch:                              10
Improvements:                            8
==============================
